# 03 — Transformação Silver dos fatos

## Objetivo

Este notebook trata e consolida as entidades transacionais do case:

- pedidos
- itens dos pedidos
- entregas
- ocorrências de atendimento

## Responsabilidades

- normalizar identificadores
- converter datas, quantidades e valores
- preservar valores originais
- sinalizar registros inválidos
- resolver duplicidades de forma determinística
- validar integridade referencial
- validar consistência financeira e temporal
- manter rastreabilidade até a camada Bronze

In [0]:
# ============================================================
# CONFIGURAÇÕES
# ============================================================

from datetime import datetime, timezone
import uuid

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.conf.set(
    "spark.sql.session.timeZone",
    "America/Sao_Paulo"
)

SCHEMA_BRONZE = "case_data_engineer_bronze"
SCHEMA_SILVER = "case_data_engineer_silver"

SILVER_FACT_RUN_ID = str(uuid.uuid4())
SILVER_FACT_TIMESTAMP = datetime.now(timezone.utc)

print(f"Silver fact run ID: {SILVER_FACT_RUN_ID}")
print(f"Timestamp UTC: {SILVER_FACT_TIMESTAMP}")

In [0]:
# ============================================================
# LOCALIZAÇÃO DINÂMICA DO CATÁLOGO
# ============================================================

catalogos_disponiveis = [
    linha["catalog"]
    for linha in spark.sql("SHOW CATALOGS").collect()
]

catalogos_ignorados = {
    "system",
    "samples",
    "hive_metastore"
}

CATALOG = None

for catalogo in catalogos_disponiveis:

    if catalogo.lower() in catalogos_ignorados:
        continue

    try:
        (
            spark.table(
                f"`{catalogo}`."
                f"`{SCHEMA_BRONZE}`."
                f"`bronze_pedidos_cabecalho`"
            )
            .limit(1)
            .collect()
        )

        CATALOG = catalogo
        break

    except Exception:
        continue

if CATALOG is None:
    raise RuntimeError(
        "A camada Bronze não foi localizada, execute primeiro o notebook 01_ingestao_bronze"
    )

spark.sql(
    f"""
    CREATE SCHEMA IF NOT EXISTS
    `{CATALOG}`.`{SCHEMA_SILVER}`
    COMMENT 'Camada Silver do case técnico de Engenharia de Dados'
    """
)

print(f"Catálogo localizado: {CATALOG}")
print(f"Bronze: {CATALOG}.{SCHEMA_BRONZE}")
print(f"Silver: {CATALOG}.{SCHEMA_SILVER}")

In [0]:
# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

CARACTERES_ACENTUADOS = (
    "ÁÀÃÂÄÉÈÊËÍÌÎÏÓÒÕÔÖÚÙÛÜÇ"
    "áàãâäéèêëíìîïóòõôöúùûüçÑñ"
)

CARACTERES_SEM_ACENTO = (
    "AAAAAEEEEIIIIOOOOOUUUUC"
    "aaaaaeeeeiiiiooooouuuucNn"
)

FORMATOS_TIMESTAMP = [
    "yyyy-MM-dd HH:mm:ss",
    "yyyy-MM-dd'T'HH:mm:ss",
    "yyyy-MM-dd'T'HH:mm:ss.SSS",
    "yyyy/MM/dd HH:mm:ss",
    "yyyy/MM/dd HH:mm",
    "dd/MM/yyyy HH:mm:ss",
    "dd/MM/yyyy HH:mm",
    "yyyy-MM-dd",
    "yyyy/MM/dd",
    "dd/MM/yyyy"
]


def remover_acentos(coluna):
    return F.translate(
        coluna.cast("string"),
        CARACTERES_ACENTUADOS,
        CARACTERES_SEM_ACENTO
    )


def normalizar_id(coluna):
    return F.upper(
        F.trim(
            coluna.cast("string")
        )
    )


def normalizar_dominio(coluna):
    valor = remover_acentos(
        F.upper(
            F.trim(
                coluna.cast("string")
            )
        )
    )

    valor = F.regexp_replace(
        valor,
        r"[^A-Z0-9]+",
        "_"
    )

    return F.regexp_replace(
        valor,
        r"(^_+|_+$)",
        ""
    )


def converter_timestamp_flexivel_coluna(coluna):
    valor = F.trim(
        coluna.cast("string")
    )

    return F.coalesce(
        *[
            F.try_to_timestamp(
                valor,
                F.lit(formato)
            )
            for formato in FORMATOS_TIMESTAMP
        ]
    )


def converter_decimal_flexivel_coluna(
    coluna,
    tipo_decimal="decimal(18,2)"
):

    valor_original = F.trim(
        coluna.cast("string")
    )

    valor_limpo = F.regexp_replace(
        valor_original,
        r"[^0-9,.\-]",
        ""
    )

    valor_padronizado = (
        F.when(
            valor_limpo.isNull()
            | (valor_limpo == ""),
            F.lit(None).cast("string")
        )
        .when(
            F.instr(valor_limpo, ",") > 0,
            F.regexp_replace(
                F.regexp_replace(
                    valor_limpo,
                    r"\.",
                    ""
                ),
                ",",
                "."
            )
        )
        .otherwise(valor_limpo)
    )

    return valor_padronizado.try_cast(
        tipo_decimal
    )


def converter_inteiro_flexivel_coluna(coluna):
    return (
        F.trim(
            coluna.cast("string")
        )
        .try_cast("long")
    )


print("Funções auxiliares disponíveis.")

In [0]:
# ============================================================
# LEITURA DAS TABELAS BRONZE
# ============================================================

bronze_pedidos = spark.table(
    f"`{CATALOG}`."
    f"`{SCHEMA_BRONZE}`."
    f"`bronze_pedidos_cabecalho`"
)

bronze_itens = spark.table(
    f"`{CATALOG}`."
    f"`{SCHEMA_BRONZE}`."
    f"`bronze_pedidos_itens`"
)

bronze_entregas = spark.table(
    f"`{CATALOG}`."
    f"`{SCHEMA_BRONZE}`."
    f"`bronze_entregas`"
)

bronze_ocorrencias = spark.table(
    f"`{CATALOG}`."
    f"`{SCHEMA_BRONZE}`."
    f"`bronze_ocorrencias`"
)

quantidades_bronze = {
    "pedidos": bronze_pedidos.count(),
    "itens": bronze_itens.count(),
    "entregas": bronze_entregas.count(),
    "ocorrencias": bronze_ocorrencias.count()
}

for entidade, quantidade in quantidades_bronze.items():
    print(f"{entidade}: {quantidade} registros")

In [0]:
# ============================================================
# VALIDAÇÃO DAS DIMENSÕES
# ============================================================

dimensoes_obrigatorias = [
    "silver_clientes",
    "silver_vendedores",
    "silver_produtos",
    "silver_canais",
    "silver_regioes"
]

tabelas_silver_existentes = {
    linha["tableName"]
    for linha in spark.sql(
        f"""
        SHOW TABLES IN
        `{CATALOG}`.`{SCHEMA_SILVER}`
        """
    ).collect()
}

dimensoes_ausentes = [
    tabela
    for tabela in dimensoes_obrigatorias
    if tabela not in tabelas_silver_existentes
]

if dimensoes_ausentes:
    raise RuntimeError(
        "As seguintes dimensões Silver não foram encontradas: "
        + ", ".join(dimensoes_ausentes)
    )

print("Dimensões Silver validadas com sucesso!")

In [0]:
# ============================================================
# INVENTÁRIO DOS ESQUEMAS TRANSACIONAIS
# ============================================================

fontes_transacionais = {
    "pedidos_cabecalho": bronze_pedidos,
    "pedidos_itens": bronze_itens,
    "entregas": bronze_entregas,
    "ocorrencias": bronze_ocorrencias
}

inventario_colunas = []

for nome_fonte, dataframe in fontes_transacionais.items():

    for posicao, campo in enumerate(
        dataframe.schema.fields,
        start=1
    ):
        inventario_colunas.append(
            (
                nome_fonte,
                posicao,
                campo.name,
                campo.dataType.simpleString(),
                campo.nullable
            )
        )

schema_inventario = """
    fonte STRING,
    posicao INT,
    coluna STRING,
    tipo STRING,
    nullable BOOLEAN
"""

df_inventario_colunas = spark.createDataFrame(
    inventario_colunas,
    schema=schema_inventario
)

display(
    df_inventario_colunas
    .orderBy("fonte", "posicao")
)

In [0]:
# ============================================================
# AMOSTRAS DAS FONTES
# ============================================================

print("PEDIDOS")
display(bronze_pedidos.limit(5))

print("ITENS")
display(bronze_itens.limit(5))

print("ENTREGAS")
display(bronze_entregas.limit(5))

print("OCORRÊNCIAS")
display(bronze_ocorrencias.limit(5))

# Relações previstas

As seguintes integridades referenciais serão avaliadas:

| Entidade | Campo | Referência |
|---|---|---|
| Pedidos | cliente | silver_clientes |
| Pedidos | vendedor | silver_vendedores |
| Pedidos | canal | silver_canais |
| Itens | pedido | silver_pedidos |
| Itens | produto | silver_produtos |
| Entregas | pedido | silver_pedidos |
| Ocorrências | pedido | silver_pedidos |


# Silver de pedidos

## Granularidade

A tabela `silver_pedidos` possui um registro por order_id.

Os registros duplicados na origem são interpretados como versões ou correções de um mesmo pedido.

A versão consolidada considerando:

1. validade dos campos críticos
2. atualização mais recente
3. completude e consistência financeira
4. desempate determinístico pelo hash do registro


Problemas de qualidade não críticos foram mantidos com indicadores `dq_*`, permitindo transparência e investigação

In [0]:
# ============================================================
# REFERÊNCIAS DIMENSIONAIS
# ============================================================

clientes_validos = (
    spark.table(
        f"`{CATALOG}`."
        f"`{SCHEMA_SILVER}`."
        f"`silver_clientes`"
    )
    .select("customer_id")
    .distinct()
    .withColumn(
        "_customer_exists",
        F.lit(True)
    )
)

vendedores_validos = (
    spark.table(
        f"`{CATALOG}`."
        f"`{SCHEMA_SILVER}`."
        f"`silver_vendedores`"
    )
    .select("seller_id")
    .distinct()
    .withColumn(
        "_seller_exists",
        F.lit(True)
    )
)

print("Referências dimensionais carregadas")

In [0]:
# ============================================================
# PREPARAÇÃO SILVER — PEDIDOS
# ============================================================

silver_pedidos_base = (
    bronze_pedidos
    .select(
        F.col("order_id").alias(
            "order_id_original"
        ),
        F.col("customer_code").alias(
            "customer_id_original"
        ),
        F.col("seller_id").alias(
            "seller_id_original"
        ),
        F.col("order_date").alias(
            "order_date_original"
        ),
        F.col("promised_date").alias(
            "promised_date_original"
        ),
        F.col("status_order").alias(
            "order_status_original"
        ),
        F.col("gross_amount").alias(
            "gross_amount_original"
        ),
        F.col("discount_amount").alias(
            "discount_amount_original"
        ),
        F.col("net_amount").alias(
            "net_amount_original"
        ),
        F.col("payment_details").alias(
            "payment_details_original"
        ),
        F.col("last_update").alias(
            "last_update_original"
        ),
        F.col("_source_record_hash"),
        F.col("_source_file"),
        F.col("_source_path"),
        F.col("_ingestion_batch_id"),
        F.col("_ingestion_timestamp_utc")
    )
    .withColumn(
        "order_id",
        normalizar_id(
            F.col("order_id_original")
        )
    )
    .withColumn(
        "customer_id",
        normalizar_id(
            F.col("customer_id_original")
        )
    )
    .withColumn(
        "seller_id",
        normalizar_id(
            F.col("seller_id_original")
        )
    )
    .withColumn(
        "order_date",
        converter_timestamp_flexivel_coluna(
            F.col("order_date_original")
        ).cast("date")
    )
    .withColumn(
        "promised_date",
        converter_timestamp_flexivel_coluna(
            F.col("promised_date_original")
        ).cast("date")
    )
    .withColumn(
        "order_status",
        normalizar_dominio(
            F.col("order_status_original")
        )
    )
    .withColumn(
        "gross_amount",
        converter_decimal_flexivel_coluna(
            F.col("gross_amount_original")
        )
    )
    .withColumn(
        "discount_amount",
        converter_decimal_flexivel_coluna(
            F.col("discount_amount_original")
        )
    )
    .withColumn(
        "net_amount",
        converter_decimal_flexivel_coluna(
            F.col("net_amount_original")
        )
    )
    .withColumn(
        "last_update",
        converter_timestamp_flexivel_coluna(
            F.col("last_update_original")
        )
    )
    .withColumn(
        "_payment_json",
        F.from_json(
            F.col("payment_details_original"),
            "priority STRING, source STRING"
        )
    )
    .withColumn(
        "payment_priority",
        normalizar_dominio(
            F.col("_payment_json.priority")
        )
    )
    .withColumn(
        "payment_source",
        normalizar_dominio(
            F.col("_payment_json.source")
        )
    )
    .join(
        clientes_validos,
        on="customer_id",
        how="left"
    )
    .join(
        vendedores_validos,
        on="seller_id",
        how="left"
    )
)

display(
    silver_pedidos_base.limit(10)
)

In [0]:
# ============================================================
# PEDIDOS - MÉTRICAS POR CHAVE DUPLICIDADE
# ============================================================

metricas_pedidos = (
    silver_pedidos_base
    .groupBy("order_id")
    .agg(
        F.count("*").alias(
            "_records_by_key"
        ),
        F.countDistinct(
            F.coalesce(
                F.col("customer_id"),
                F.lit("__NULL__")
            )
        ).alias(
            "_distinct_customers"
        ),
        F.countDistinct(
            F.coalesce(
                F.col("seller_id"),
                F.lit("__NULL__")
            )
        ).alias(
            "_distinct_sellers"
        ),
        F.countDistinct(
            F.coalesce(
                F.col("order_date").cast("string"),
                F.lit("__NULL__")
            )
        ).alias(
            "_distinct_order_dates"
        ),
        F.countDistinct(
            F.coalesce(
                F.col("order_status"),
                F.lit("__NULL__")
            )
        ).alias(
            "_distinct_statuses"
        ),
        F.countDistinct(
            F.coalesce(
                F.col("gross_amount").cast("string"),
                F.lit("__NULL__")
            )
        ).alias(
            "_distinct_gross_amounts"
        ),
        F.countDistinct(
            F.coalesce(
                F.col("net_amount").cast("string"),
                F.lit("__NULL__")
            )
        ).alias(
            "_distinct_net_amounts"
        )
    )
)

display(
    metricas_pedidos
    .filter(F.col("_records_by_key") > 1)
    .orderBy("order_id")
)

In [0]:
# ============================================================
# PEDIDOS - QUALIDADE
# ============================================================

STATUS_PEDIDO_VALIDOS = [
    "FATURADO",
    "ENTREGUE",
    "CANCELADO",
    "EM_SEPARACAO"
]


silver_pedidos_preparacao = (
    silver_pedidos_base
    .join(
        metricas_pedidos,
        on="order_id",
        how="left"
    )

    # Chaves e relacionamentos
    .withColumn(
        "dq_missing_order_id",
        F.col("order_id").isNull()
        | (F.col("order_id") == "")
    )
    .withColumn(
        "dq_missing_customer",
        F.col("customer_id").isNull()
        | (F.col("customer_id") == "")
    )
    .withColumn(
        "dq_orphan_customer",
        F.col("customer_id").isNotNull()
        & (F.col("customer_id") != "")
        & F.col("_customer_exists").isNull()
    )
    .withColumn(
        "dq_missing_seller",
        F.col("seller_id").isNull()
        | (F.col("seller_id") == "")
    )
    .withColumn(
        "dq_orphan_seller",
        F.col("seller_id").isNotNull()
        & (F.col("seller_id") != "")
        & F.col("_seller_exists").isNull()
    )

    # Datas
    .withColumn(
        "dq_missing_order_date",
        F.col("order_date_original").isNull()
        | (
            F.trim(
                F.col("order_date_original")
            ) == ""
        )
    )
    .withColumn(
        "dq_invalid_order_date",
        ~F.col("dq_missing_order_date")
        & F.col("order_date").isNull()
    )
    .withColumn(
        "dq_missing_promised_date",
        F.col("promised_date_original").isNull()
        | (
            F.trim(
                F.col("promised_date_original")
            ) == ""
        )
    )
    .withColumn(
        "dq_invalid_promised_date",
        ~F.col("dq_missing_promised_date")
        & F.col("promised_date").isNull()
    )
    .withColumn(
        "dq_promised_before_order",
        F.col("order_date").isNotNull()
        & F.col("promised_date").isNotNull()
        & (
            F.col("promised_date")
            < F.col("order_date")
        )
    )
    .withColumn(
        "dq_missing_last_update",
        F.col("last_update_original").isNull()
        | (
            F.trim(
                F.col("last_update_original")
            ) == ""
        )
    )
    .withColumn(
        "dq_invalid_last_update",
        ~F.col("dq_missing_last_update")
        & F.col("last_update").isNull()
    )

    # Status
    .withColumn(
        "dq_missing_status",
        F.col("order_status").isNull()
        | (F.col("order_status") == "")
    )
    .withColumn(
        "dq_unexpected_status",
        ~F.col("dq_missing_status")
        & (
            ~F.col("order_status").isin(
                STATUS_PEDIDO_VALIDOS
            )
        )
    )

    # Valores
    .withColumn(
        "dq_missing_gross_amount",
        F.col("gross_amount_original").isNull()
        | (
            F.trim(
                F.col("gross_amount_original")
            ) == ""
        )
    )
    .withColumn(
        "dq_invalid_gross_amount",
        ~F.col("dq_missing_gross_amount")
        & F.col("gross_amount").isNull()
    )
    .withColumn(
        "dq_missing_discount_amount",
        F.col("discount_amount_original").isNull()
        | (
            F.trim(
                F.col("discount_amount_original")
            ) == ""
        )
    )
    .withColumn(
        "dq_invalid_discount_amount",
        ~F.col("dq_missing_discount_amount")
        & F.col("discount_amount").isNull()
    )
    .withColumn(
        "dq_missing_net_amount",
        F.col("net_amount_original").isNull()
        | (
            F.trim(
                F.col("net_amount_original")
            ) == ""
        )
    )
    .withColumn(
        "dq_invalid_net_amount",
        ~F.col("dq_missing_net_amount")
        & F.col("net_amount").isNull()
    )
    .withColumn(
        "dq_nonpositive_gross_amount",
        F.col("gross_amount").isNotNull()
        & (F.col("gross_amount") <= 0)
    )
    .withColumn(
        "dq_negative_discount_amount",
        F.col("discount_amount").isNotNull()
        & (F.col("discount_amount") < 0)
    )
    .withColumn(
        "dq_negative_net_amount",
        F.col("net_amount").isNotNull()
        & (F.col("net_amount") < 0)
    )

    # Consistência financeira
    .withColumn(
        "dq_financial_not_evaluable",
        F.col("gross_amount").isNull()
        | F.col("discount_amount").isNull()
        | F.col("net_amount").isNull()
    )
    .withColumn(
        "financial_difference",
        (
            F.col("gross_amount")
            - F.col("discount_amount")
            - F.col("net_amount")
        ).cast("decimal(18,2)")
    )
    .withColumn(
        "dq_financial_mismatch",
        ~F.col("dq_financial_not_evaluable")
        & (
            F.abs(
                F.col("financial_difference")
            ) > F.lit(0.01)
        )
    )

    # JSON do pagamento
    .withColumn(
        "dq_invalid_payment_json",
        F.col("payment_details_original").isNotNull()
        & (
            F.trim(
                F.col("payment_details_original")
            ) != ""
        )
        & F.col("_payment_json").isNull()
    )

    # Duplicidade
    .withColumn(
        "dq_duplicate_key",
        F.col("_records_by_key") > 1
    )
    .withColumn(
        "dq_conflicting_duplicate",
        (
            F.col("_distinct_customers") > 1
        )
        |
        (
            F.col("_distinct_sellers") > 1
        )
        |
        (
            F.col("_distinct_order_dates") > 1
        )
        |
        (
            F.col("_distinct_statuses") > 1
        )
        |
        (
            F.col("_distinct_gross_amounts") > 1
        )
        |
        (
            F.col("_distinct_net_amounts") > 1
        )
    )
)

In [0]:
# ============================================================
# PEDIDOS - PRIORIZAÇÃO E DEDUPLICAÇÃO
# ============================================================

silver_pedidos_preparacao = (
    silver_pedidos_preparacao
    .withColumn(
        "_critical_record_valid",
        ~F.col("dq_missing_order_date")
        & ~F.col("dq_invalid_order_date")
        & ~F.col("dq_missing_promised_date")
        & ~F.col("dq_invalid_promised_date")
        & ~F.col("dq_missing_gross_amount")
        & ~F.col("dq_invalid_gross_amount")
        & ~F.col("dq_missing_discount_amount")
        & ~F.col("dq_invalid_discount_amount")
        & ~F.col("dq_missing_net_amount")
        & ~F.col("dq_invalid_net_amount")
    )
    .withColumn(
        "_quality_score",
        F.when(
            ~F.col("dq_missing_order_date")
            & ~F.col("dq_invalid_order_date"),
            15
        ).otherwise(0)
        +
        F.when(
            ~F.col("dq_missing_promised_date")
            & ~F.col("dq_invalid_promised_date"),
            10
        ).otherwise(0)
        +
        F.when(
            ~F.col("dq_missing_gross_amount")
            & ~F.col("dq_invalid_gross_amount"),
            15
        ).otherwise(0)
        +
        F.when(
            ~F.col("dq_missing_discount_amount")
            & ~F.col("dq_invalid_discount_amount"),
            5
        ).otherwise(0)
        +
        F.when(
            ~F.col("dq_missing_net_amount")
            & ~F.col("dq_invalid_net_amount"),
            15
        ).otherwise(0)
        +
        F.when(
            ~F.col("dq_financial_not_evaluable")
            & ~F.col("dq_financial_mismatch"),
            15
        ).otherwise(0)
        +
        F.when(
            F.col("_customer_exists") == True,
            10
        ).otherwise(0)
        +
        F.when(
            F.col("_seller_exists") == True,
            10
        ).otherwise(0)
        +
        F.when(
            ~F.col("dq_missing_status")
            & ~F.col("dq_unexpected_status"),
            10
        ).otherwise(0)
        +
        F.when(
            ~F.col("dq_missing_last_update")
            & ~F.col("dq_invalid_last_update"),
            5
        ).otherwise(0)
    )
)


janela_prioridade_pedido = (
    Window
    .partitionBy("order_id")
    .orderBy(
        F.col(
            "_critical_record_valid"
        ).desc(),
        F.col(
            "last_update"
        ).desc_nulls_last(),
        F.col(
            "_quality_score"
        ).desc(),
        F.col(
            "_ingestion_timestamp_utc"
        ).desc(),
        F.col(
            "_source_record_hash"
        ).asc()
    )
)


silver_pedidos = (
    silver_pedidos_preparacao
    .withColumn(
        "_deduplication_rank",
        F.row_number().over(
            janela_prioridade_pedido
        )
    )
    .filter(
        F.col("_deduplication_rank") == 1
    )
    .withColumn(
        "_silver_fact_run_id",
        F.lit(SILVER_FACT_RUN_ID)
    )
    .withColumn(
        "_silver_processed_at_utc",
        F.lit(SILVER_FACT_TIMESTAMP)
    )
    .drop(
        "_payment_json",
        "_customer_exists",
        "_seller_exists",
        "_records_by_key",
        "_distinct_customers",
        "_distinct_sellers",
        "_distinct_order_dates",
        "_distinct_statuses",
        "_distinct_gross_amounts",
        "_distinct_net_amounts",
        "_critical_record_valid",
        "_quality_score",
        "_deduplication_rank"
    )
)

display(
    silver_pedidos
    .orderBy("order_id")
)

In [0]:
# ============================================================
# PEDIDOS - VALIDAÇÃO
# ============================================================

quantidade_pedidos_bronze = (
    bronze_pedidos.count()
)

quantidade_pedidos_distintos_esperada = (
    bronze_pedidos
    .select(
        normalizar_id(
            F.col("order_id")
        ).alias("order_id")
    )
    .distinct()
    .count()
)

quantidade_pedidos_silver = (
    silver_pedidos.count()
)

duplicidades_pedidos_silver = (
    silver_pedidos
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

chaves_nulas_pedidos = (
    silver_pedidos
    .filter(
        F.col("order_id").isNull()
        | (F.col("order_id") == "")
    )
    .count()
)

pedidos_clientes_orfaos = (
    silver_pedidos
    .filter(F.col("dq_orphan_customer"))
    .count()
)

pedidos_vendedores_orfaos = (
    silver_pedidos
    .filter(F.col("dq_orphan_seller"))
    .count()
)

pedidos_divergencia_financeira = (
    silver_pedidos
    .filter(F.col("dq_financial_mismatch"))
    .count()
)

pedidos_status_ausente = (
    silver_pedidos
    .filter(F.col("dq_missing_status"))
    .count()
)

print(f"Bronze: {quantidade_pedidos_bronze}")

print(
    f"Chaves distintas esperadas: "
    f"{quantidade_pedidos_distintos_esperada}"
)

print(f"Silver: {quantidade_pedidos_silver}")

print(
    f"Registros consolidados: "
    f"{quantidade_pedidos_bronze - quantidade_pedidos_silver}"
)

print(
    f"Duplicidades na Silver: "
    f"{duplicidades_pedidos_silver}"
)

print(
    f"Chaves nulas na Silver: "
    f"{chaves_nulas_pedidos}"
)

print(
    f"Pedidos com cliente órfão: "
    f"{pedidos_clientes_orfaos}"
)

print(
    f"Pedidos com vendedor órfão: "
    f"{pedidos_vendedores_orfaos}"
)

print(
    f"Divergências financeiras: "
    f"{pedidos_divergencia_financeira}"
)

print(
    f"Pedidos sem status: "
    f"{pedidos_status_ausente}"
)


if (
    quantidade_pedidos_silver
    != quantidade_pedidos_distintos_esperada
):
    raise RuntimeError(
        "A quantidade da Silver de pedidos não corresponde às chaves distintas da Bronze"
    )

if duplicidades_pedidos_silver > 0:
    raise RuntimeError(
        "A Silver de pedidos ainda possui duplicidades"
    )

if chaves_nulas_pedidos > 0:
    raise RuntimeError(
        "A Silver de pedidos possui chaves nulas"
    )

In [0]:
# ============================================================
# PEDIDOS - CHECK DOS DUPLICADOS
# ============================================================

display(
    silver_pedidos
    .filter(F.col("dq_duplicate_key"))
    .select(
        "order_id",
        "customer_id",
        "seller_id",
        "order_date",
        "promised_date",
        "order_status",
        "gross_amount",
        "discount_amount",
        "net_amount",
        "last_update",
        "dq_financial_mismatch",
        "dq_conflicting_duplicate"
    )
    .orderBy("order_id")
)

In [0]:
# ============================================================
# SILVER — PEDIDOS
# ============================================================

TABELA_SILVER_PEDIDOS = (
    f"`{CATALOG}`."
    f"`{SCHEMA_SILVER}`."
    f"`silver_pedidos`"
)

(
    silver_pedidos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_SILVER_PEDIDOS)
)

quantidade_pedidos_persistidos = (
    spark.table(TABELA_SILVER_PEDIDOS)
    .count()
)

if (
    quantidade_pedidos_persistidos
    != quantidade_pedidos_silver
):
    raise RuntimeError(
        "A quantidade persistida de pedidos diverge da quantidade do DataFrame Silver"
    )

print(
    f"Tabela criada: "
    f"{CATALOG}.{SCHEMA_SILVER}.silver_pedidos"
)

print(
    f"Registros persistidos: "
    f"{quantidade_pedidos_persistidos}"
)

In [0]:
# ============================================================
# REFERÊNCIAS — PEDIDOS E PRODUTOS
# ============================================================

pedidos_validos = (
    spark.table(
        f"`{CATALOG}`."
        f"`{SCHEMA_SILVER}`."
        f"`silver_pedidos`"
    )
    .select("order_id")
    .distinct()
    .withColumn(
        "_order_exists",
        F.lit(True)
    )
)

produtos_validos = (
    spark.table(
        f"`{CATALOG}`."
        f"`{SCHEMA_SILVER}`."
        f"`silver_produtos`"
    )
    .select("product_id")
    .distinct()
    .withColumn(
        "_product_exists",
        F.lit(True)
    )
)

print("Referências de pedidos e produtos carregadas.")

In [0]:
# ============================================================
# REFERÊNCIAS — PEDIDOS E PRODUTOS
# ============================================================

pedidos_validos = (
    spark.table(
        f"`{CATALOG}`."
        f"`{SCHEMA_SILVER}`."
        f"`silver_pedidos`"
    )
    .select("order_id")
    .distinct()
    .withColumn(
        "_order_exists",
        F.lit(True)
    )
)

produtos_validos = (
    spark.table(
        f"`{CATALOG}`."
        f"`{SCHEMA_SILVER}`."
        f"`silver_produtos`"
    )
    .select("product_id")
    .distinct()
    .withColumn(
        "_product_exists",
        F.lit(True)
    )
)

print("Referências de pedidos e produtos carregadas!")

In [0]:
# ============================================================
# ESQUEMA SILVER — ITENS
# ============================================================

# Localiza uma coluna da origem considerando possíveis aliases
def localizar_coluna(
    dataframe,
    candidatos,
    nome_logico
):
    mapa_colunas = {
        coluna.lower().strip(): coluna
        for coluna in dataframe.columns
    }

    for candidato in candidatos:
        candidato_normalizado = candidato.lower().strip()

        if candidato_normalizado in mapa_colunas:
            return mapa_colunas[candidato_normalizado]

    raise RuntimeError(
        f"Não foi possível localizar o campo lógico "
        f"'{nome_logico}'.\n"
        f"Candidatos avaliados: {candidatos}\n"
        f"Colunas disponíveis: {dataframe.columns}"
    )


# ------------------------------------------------------------
# DINÂMICA DAS COLUNAS
# ------------------------------------------------------------

COL_ORDER_ID = localizar_coluna(
    bronze_itens,
    [
        "order_id",
        "order_ref",
        "pedido_id",
        "id_pedido"
    ],
    "identificador do pedido"
)

COL_ITEM_SEQ = localizar_coluna(
    bronze_itens,
    [
        "item_seq",
        "item_sequence",
        "sequence",
        "seq_item"
    ],
    "sequência do item"
)

COL_PRODUCT_ID = localizar_coluna(
    bronze_itens,
    [
        "product_id",
        "product_code",
        "sku",
        "sku_id",
        "codigo_produto",
        "produto_id"
    ],
    "identificador do produto"
)

COL_QUANTITY = localizar_coluna(
    bronze_itens,
    [
        "quantity",
        "qty",
        "quantidade"
    ],
    "quantidade"
)

COL_UNIT_PRICE = localizar_coluna(
    bronze_itens,
    [
        "unit_price",
        "price",
        "preco_unitario",
        "valor_unitario"
    ],
    "preço unitário"
)

COL_TOTAL_ITEM = localizar_coluna(
    bronze_itens,
    [
        "total_item",
        "item_total",
        "total_amount",
        "valor_total"
    ],
    "total do item"
)


print("Mapeamento de itens resolvido:")

for campo_logico, campo_origem in {
    "order_id": COL_ORDER_ID,
    "item_seq": COL_ITEM_SEQ,
    "product_id": COL_PRODUCT_ID,
    "quantity": COL_QUANTITY,
    "unit_price": COL_UNIT_PRICE,
    "total_item": COL_TOTAL_ITEM
}.items():

    print(
        f"{campo_logico:<15} -> {campo_origem}"
    )


# ------------------------------------------------------------
# REFERÊNCIAS SILVER
# ------------------------------------------------------------

pedidos_validos = (
    spark.table(
        f"`{CATALOG}`."
        f"`{SCHEMA_SILVER}`."
        f"`silver_pedidos`"
    )
    .select("order_id")
    .distinct()
    .withColumn(
        "_order_exists",
        F.lit(True)
    )
)

produtos_validos = (
    spark.table(
        f"`{CATALOG}`."
        f"`{SCHEMA_SILVER}`."
        f"`silver_produtos`"
    )
    .select("product_id")
    .distinct()
    .withColumn(
        "_product_exists",
        F.lit(True)
    )
)


# ------------------------------------------------------------
# SILVER — ITENS DOS PEDIDOS
# ------------------------------------------------------------

silver_itens_base = (
    bronze_itens
    .select(
        F.col(COL_ORDER_ID).alias(
            "order_id_original"
        ),
        F.col(COL_ITEM_SEQ).alias(
            "item_seq_original"
        ),
        F.col(COL_PRODUCT_ID).alias(
            "product_id_original"
        ),
        F.col(COL_QUANTITY).alias(
            "quantity_original"
        ),
        F.col(COL_UNIT_PRICE).alias(
            "unit_price_original"
        ),
        F.col(COL_TOTAL_ITEM).alias(
            "total_item_original"
        ),
        F.col("_source_record_hash"),
        F.col("_source_file"),
        F.col("_source_path"),
        F.col("_ingestion_batch_id"),
        F.col("_ingestion_timestamp_utc")
    )
    .withColumn(
        "order_id",
        normalizar_id(
            F.col("order_id_original")
        )
    )
    .withColumn(
        "item_seq",
        converter_inteiro_flexivel_coluna(
            F.col("item_seq_original")
        )
    )
    .withColumn(
        "product_id",
        normalizar_id(
            F.col("product_id_original")
        )
    )
    .withColumn(
        "quantity",
        converter_decimal_flexivel_coluna(
            F.col("quantity_original"),
            "decimal(18,4)"
        )
    )
    .withColumn(
        "unit_price",
        converter_decimal_flexivel_coluna(
            F.col("unit_price_original"),
            "decimal(18,2)"
        )
    )
    .withColumn(
        "total_item",
        converter_decimal_flexivel_coluna(
            F.col("total_item_original"),
            "decimal(18,2)"
        )
    )
    .withColumn(
        "calculated_total_item",
        (
            F.col("quantity")
            * F.col("unit_price")
        ).cast("decimal(18,2)")
    )
    .withColumn(
        "item_total_difference",
        (
            F.col("total_item")
            - F.col("calculated_total_item")
        ).cast("decimal(18,2)")
    )
    .withColumn(
        "item_business_key",
        F.when(
            F.col("order_id").isNotNull()
            & F.col("item_seq").isNotNull(),
            F.concat(
                F.col("order_id"),
                F.lit("|"),
                F.lpad(
                    F.col("item_seq").cast("string"),
                    4,
                    "0"
                )
            )
        )
    )
    .join(
        pedidos_validos,
        on="order_id",
        how="left"
    )
    .join(
        produtos_validos,
        on="product_id",
        how="left"
    )
)


quantidade_silver_itens_base = (
    silver_itens_base.count()
)

print(
    f"\nsilver_itens_base criada com "
    f"{quantidade_silver_itens_base} registros."
)

display(
    silver_itens_base.limit(10)
)

In [0]:
# ============================================================
# MÉTRICAS DE DUPLICIDADE — ITENS DOS PEDIDOS
# ============================================================

metricas_itens = (
    silver_itens_base
    .groupBy(
        "order_id",
        "item_seq"
    )
    .agg(
        F.count("*").alias(
            "_records_by_key"
        ),
        F.countDistinct(
            F.coalesce(
                F.col("product_id"),
                F.lit("__NULL__")
            )
        ).alias(
            "_distinct_products"
        ),
        F.countDistinct(
            F.coalesce(
                F.col("quantity").cast("string"),
                F.lit("__NULL__")
            )
        ).alias(
            "_distinct_quantities"
        ),
        F.countDistinct(
            F.coalesce(
                F.col("unit_price").cast("string"),
                F.lit("__NULL__")
            )
        ).alias(
            "_distinct_unit_prices"
        ),
        F.countDistinct(
            F.coalesce(
                F.col("total_item").cast("string"),
                F.lit("__NULL__")
            )
        ).alias(
            "_distinct_total_items"
        )
    )
)

print("Chaves duplicadas identificadas:")

display(
    metricas_itens
    .filter(
        F.col("_records_by_key") > 1
    )
    .orderBy(
        "order_id",
        "item_seq"
    )
)

In [0]:
# ============================================================
# QUALIDADE — ITENS DOS PEDIDOS
# ============================================================

silver_itens_preparacao = (
    silver_itens_base
    .join(
        metricas_itens,
        on=[
            "order_id",
            "item_seq"
        ],
        how="left"
    )

    # --------------------------------------------------------
    # Chave do pedido
    # --------------------------------------------------------
    .withColumn(
        "dq_missing_order_id",
        F.col("order_id").isNull()
        | (F.col("order_id") == "")
    )
    .withColumn(
        "dq_orphan_order",
        F.col("order_id").isNotNull()
        & (F.col("order_id") != "")
        & F.col("_order_exists").isNull()
    )

    # --------------------------------------------------------
    # Sequência do item
    # --------------------------------------------------------
    .withColumn(
        "dq_missing_item_seq",
        F.col("item_seq_original").isNull()
        | (
            F.trim(
                F.col("item_seq_original").cast("string")
            ) == ""
        )
    )
    .withColumn(
        "dq_invalid_item_seq",
        ~F.col("dq_missing_item_seq")
        & F.col("item_seq").isNull()
    )
    .withColumn(
        "dq_nonpositive_item_seq",
        F.col("item_seq").isNotNull()
        & (F.col("item_seq") <= 0)
    )

    # --------------------------------------------------------
    # Produto
    # --------------------------------------------------------
    .withColumn(
        "dq_missing_product",
        F.col("product_id").isNull()
        | (F.col("product_id") == "")
    )
    .withColumn(
        "dq_orphan_product",
        F.col("product_id").isNotNull()
        & (F.col("product_id") != "")
        & F.col("_product_exists").isNull()
    )

    # --------------------------------------------------------
    # Quantidade
    # --------------------------------------------------------
    .withColumn(
        "dq_missing_quantity",
        F.col("quantity_original").isNull()
        | (
            F.trim(
                F.col("quantity_original").cast("string")
            ) == ""
        )
    )
    .withColumn(
        "dq_invalid_quantity",
        ~F.col("dq_missing_quantity")
        & F.col("quantity").isNull()
    )
    .withColumn(
        "dq_nonpositive_quantity",
        F.col("quantity").isNotNull()
        & (F.col("quantity") <= 0)
    )

    # --------------------------------------------------------
    # Preço unitário
    # --------------------------------------------------------
    .withColumn(
        "dq_missing_unit_price",
        F.col("unit_price_original").isNull()
        | (
            F.trim(
                F.col("unit_price_original").cast("string")
            ) == ""
        )
    )
    .withColumn(
        "dq_invalid_unit_price",
        ~F.col("dq_missing_unit_price")
        & F.col("unit_price").isNull()
    )
    .withColumn(
        "dq_negative_unit_price",
        F.col("unit_price").isNotNull()
        & (F.col("unit_price") < 0)
    )

    # --------------------------------------------------------
    # Total informado
    # --------------------------------------------------------
    .withColumn(
        "dq_missing_total_item",
        F.col("total_item_original").isNull()
        | (
            F.trim(
                F.col("total_item_original").cast("string")
            ) == ""
        )
    )
    .withColumn(
        "dq_invalid_total_item",
        ~F.col("dq_missing_total_item")
        & F.col("total_item").isNull()
    )
    .withColumn(
        "dq_negative_total_item",
        F.col("total_item").isNotNull()
        & (F.col("total_item") < 0)
    )

    # --------------------------------------------------------
    # Consistência quantity × unit_price
    # --------------------------------------------------------
    .withColumn(
        "dq_item_total_not_evaluable",
        F.col("quantity").isNull()
        | F.col("unit_price").isNull()
        | F.col("total_item").isNull()
    )
    .withColumn(
        "dq_item_total_mismatch",
        ~F.col("dq_item_total_not_evaluable")
        & (
            F.abs(
                F.col("item_total_difference")
            ) > F.lit(0.01)
        )
    )

    # --------------------------------------------------------
    # Duplicidades
    # --------------------------------------------------------
    .withColumn(
        "dq_duplicate_key",
        F.col("_records_by_key") > 1
    )
    .withColumn(
        "dq_conflicting_duplicate",
        (
            F.col("_distinct_products") > 1
        )
        |
        (
            F.col("_distinct_quantities") > 1
        )
        |
        (
            F.col("_distinct_unit_prices") > 1
        )
        |
        (
            F.col("_distinct_total_items") > 1
        )
    )
)

In [0]:
# ============================================================
# PONTUAÇÃO DE QUALIDADE — ITENS
# ============================================================

silver_itens_preparacao = (
    silver_itens_preparacao
    .withColumn(
        "_quality_score",

        # Pedido válido
        F.when(
            F.col("_order_exists") == True,
            25
        ).otherwise(0)

        +

        # Produto válido
        F.when(
            F.col("_product_exists") == True,
            25
        ).otherwise(0)

        +

        # Sequência válida
        F.when(
            ~F.col("dq_missing_item_seq")
            & ~F.col("dq_invalid_item_seq")
            & ~F.col("dq_nonpositive_item_seq"),
            10
        ).otherwise(0)

        +

        # Quantidade válida
        F.when(
            ~F.col("dq_missing_quantity")
            & ~F.col("dq_invalid_quantity")
            & ~F.col("dq_nonpositive_quantity"),
            10
        ).otherwise(0)

        +

        # Preço válido
        F.when(
            ~F.col("dq_missing_unit_price")
            & ~F.col("dq_invalid_unit_price")
            & ~F.col("dq_negative_unit_price"),
            10
        ).otherwise(0)

        +

        # Total válido
        F.when(
            ~F.col("dq_missing_total_item")
            & ~F.col("dq_invalid_total_item")
            & ~F.col("dq_negative_total_item"),
            10
        ).otherwise(0)

        +

        # Consistência matemática
        F.when(
            ~F.col("dq_item_total_not_evaluable")
            & ~F.col("dq_item_total_mismatch"),
            20
        ).otherwise(0)
    )
)

In [0]:
# ============================================================
# DEDUPLICAÇÃO — ITENS
# ============================================================

janela_prioridade_item = (
    Window
    .partitionBy(
        "order_id",
        "item_seq"
    )
    .orderBy(
        F.col("_quality_score").desc(),
        F.col(
            "_ingestion_timestamp_utc"
        ).desc(),
        F.col(
            "_source_record_hash"
        ).asc()
    )
)

silver_pedidos_itens = (
    silver_itens_preparacao
    .withColumn(
        "_deduplication_rank",
        F.row_number().over(
            janela_prioridade_item
        )
    )
    .filter(
        F.col("_deduplication_rank") == 1
    )
    .withColumn(
        "_silver_fact_run_id",
        F.lit(SILVER_FACT_RUN_ID)
    )
    .withColumn(
        "_silver_processed_at_utc",
        F.lit(SILVER_FACT_TIMESTAMP)
    )
    .drop(
        "_order_exists",
        "_product_exists",
        "_records_by_key",
        "_distinct_products",
        "_distinct_quantities",
        "_distinct_unit_prices",
        "_distinct_total_items",
        "_quality_score",
        "_deduplication_rank"
    )
)

display(
    silver_pedidos_itens
    .orderBy(
        "order_id",
        "item_seq"
    )
)

In [0]:
# ============================================================
# VALIDAÇÃO — ITENS DOS PEDIDOS
# ============================================================

quantidade_itens_bronze = (
    bronze_itens.count()
)

quantidade_itens_distintos_esperada = (
    bronze_itens
    .select(
        normalizar_id(
            F.col(COL_ORDER_ID)
        ).alias(
            "order_id"
        ),
        converter_inteiro_flexivel_coluna(
            F.col(COL_ITEM_SEQ)
        ).alias(
            "item_seq"
        )
    )
    .distinct()
    .count()
)

quantidade_itens_silver = (
    silver_pedidos_itens.count()
)

duplicidades_itens_silver = (
    silver_pedidos_itens
    .groupBy(
        "order_id",
        "item_seq"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

chaves_nulas_itens = (
    silver_pedidos_itens
    .filter(
        F.col("order_id").isNull()
        | (F.col("order_id") == "")
        | F.col("item_seq").isNull()
    )
    .count()
)

itens_pedidos_orfaos = (
    silver_pedidos_itens
    .filter(
        F.col("dq_orphan_order") == True
    )
    .count()
)

itens_produtos_orfaos = (
    silver_pedidos_itens
    .filter(
        F.col("dq_orphan_product") == True
    )
    .count()
)

itens_divergencia_total = (
    silver_pedidos_itens
    .filter(
        F.col("dq_item_total_mismatch") == True
    )
    .count()
)

itens_total_nao_avaliavel = (
    silver_pedidos_itens
    .filter(
        F.col("dq_item_total_not_evaluable") == True
    )
    .count()
)

print(
    f"Bronze: "
    f"{quantidade_itens_bronze}"
)

print(
    f"Chaves distintas esperadas: "
    f"{quantidade_itens_distintos_esperada}"
)

print(
    f"Silver: "
    f"{quantidade_itens_silver}"
)

print(
    f"Registros consolidados: "
    f"{quantidade_itens_bronze - quantidade_itens_silver}"
)

print(
    f"Duplicidades na Silver: "
    f"{duplicidades_itens_silver}"
)

print(
    f"Chaves nulas na Silver: "
    f"{chaves_nulas_itens}"
)

print(
    f"Itens com pedido órfão: "
    f"{itens_pedidos_orfaos}"
)

print(
    f"Itens com produto órfão: "
    f"{itens_produtos_orfaos}"
)

print(
    f"Divergências quantity × unit_price: "
    f"{itens_divergencia_total}"
)

print(
    f"Totais não avaliáveis: "
    f"{itens_total_nao_avaliavel}"
)

if (
    quantidade_itens_silver
    != quantidade_itens_distintos_esperada
):
    raise RuntimeError(
        "A quantidade da Silver de itens não corresponde as chaves compostas distintas da Bronze"
    )

if duplicidades_itens_silver > 0:
    raise RuntimeError(
        "A Silver de itens ainda possui duplicidades"
    )

if chaves_nulas_itens > 0:
    raise RuntimeError(
        "A Silver de itens possui chaves nulas"
    )

In [0]:
# ============================================================
# CONFERÊNCIA DAS DIVERGÊNCIAS DOS ITENS
# ============================================================

display(
    silver_pedidos_itens
    .filter(
        F.col("dq_item_total_mismatch") == True
    )
    .select(
        "order_id",
        "item_seq",
        "product_id",
        "quantity",
        "unit_price",
        "calculated_total_item",
        "total_item",
        "item_total_difference"
    )
    .orderBy(
        F.abs(
            F.col("item_total_difference")
        ).desc()
    )
)

In [0]:
# ============================================================
# CONFERÊNCIA DOS PRODUTOS UNICOS
# ============================================================

display(
    silver_pedidos_itens
    .filter(
        F.col("dq_orphan_product") == True
    )
    .select(
        "order_id",
        "item_seq",
        "product_id",
        "quantity",
        "unit_price",
        "total_item"
    )
    .orderBy(
        "order_id",
        "item_seq"
    )
)

In [0]:
# ============================================================
# SILVER — ITENS DOS PEDIDOS
# ============================================================

TABELA_SILVER_PEDIDOS_ITENS = (
    f"`{CATALOG}`."
    f"`{SCHEMA_SILVER}`."
    f"`silver_pedidos_itens`"
)

(
    silver_pedidos_itens.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        TABELA_SILVER_PEDIDOS_ITENS
    )
)

quantidade_itens_persistidos = (
    spark.table(
        TABELA_SILVER_PEDIDOS_ITENS
    )
    .count()
)

if (
    quantidade_itens_persistidos
    != quantidade_itens_silver
):
    raise RuntimeError(
        "A quantidade persistida de itens diverge da quantidade do DataFrame Silver!"
    )

print(
    f"Tabela criada: "
    f"{CATALOG}.{SCHEMA_SILVER}."
    f"silver_pedidos_itens"
)

print(
    f"Registros persistidos: "
    f"{quantidade_itens_persistidos}"
)

# Silver de entregas

## Granularidade

A tabela silver_entregas possui um registro consolidado por pedido.

Duplicidades são interpretadas como atualizações do processo logístico onde a prioridade:

1. pedido existente
2. datas e custo válidos
3. consistência entre status e timestamps
4. maior avanço temporal da entrega
5. desempate determinístico pelos metadados da ingestão

Problemas de qualidade não críticos foram mantidos com indicadores `dq_*`, permitindo transparência e investigação

In [0]:
# ============================================================
# RESOLUÇÃO DO ESQUEMA — ENTREGAS
# ============================================================

# Localiza campos simples ou aninhados no DataFrame.
def localizar_campo_spark(
    dataframe,
    candidatos,
    nome_logico,
    obrigatorio=True
):

    for candidato in candidatos:
        try:
            dataframe.select(
                F.col(candidato)
            ).limit(0).collect()

            return candidato

        except Exception:
            continue

    if obrigatorio:
        raise RuntimeError(
            f"Não foi possível localizar o campo lógico "
            f"'{nome_logico}'. "
            f"Candidatos avaliados: {candidatos}. "
            f"Colunas disponíveis: {dataframe.columns}"
        )

    return None


COL_DELIVERY_ID = localizar_campo_spark(
    bronze_entregas,
    [
        "delivery_id",
        "shipment_id",
        "entrega_id",
        "id_entrega"
    ],
    "identificador da entrega"
)

COL_DELIVERY_ORDER_ID = localizar_campo_spark(
    bronze_entregas,
    [
        "order_ref",
        "order_id",
        "pedido_id",
        "id_pedido"
    ],
    "identificador do pedido"
)

COL_DELIVERY_CARRIER = localizar_campo_spark(
    bronze_entregas,
    [
        "carrier",
        "carrier_name",
        "transportadora"
    ],
    "transportadora"
)

COL_DELIVERY_SERVICE_LEVEL = localizar_campo_spark(
    bronze_entregas,
    [
        "service_level",
        "shipping_method",
        "delivery_method",
        "modalidade"
    ],
    "nível de serviço",
    obrigatorio=False
)

COL_DELIVERY_STATUS = localizar_campo_spark(
    bronze_entregas,
    [
        "delivery_status",
        "status",
        "shipping_status"
    ],
    "status da entrega"
)

COL_DELIVERY_SHIPPED_AT = localizar_campo_spark(
    bronze_entregas,
    [
        "timestamps.shipped_at",
        "shipped_at",
        "shipment_date"
    ],
    "data de envio"
)

COL_DELIVERY_DELIVERED_AT = localizar_campo_spark(
    bronze_entregas,
    [
        "timestamps.delivered_at",
        "delivered_at",
        "delivery_date"
    ],
    "data de entrega"
)

COL_DELIVERY_COST = localizar_campo_spark(
    bronze_entregas,
    [
        "cost",
        "freight_cost",
        "shipping_cost",
        "delivery_cost"
    ],
    "custo da entrega"
)

COL_DELIVERY_TRACKING = localizar_campo_spark(
    bronze_entregas,
    [
        "tracking_code",
        "tracking_id",
        "tracking_number",
        "tracking"
    ],
    "código de rastreamento",
    obrigatorio=False
)

COL_DELIVERY_DESTINATION = localizar_campo_spark(
    bronze_entregas,
    [
        "destination"
    ],
    "destino",
    obrigatorio=False
)

mapeamento_entregas = {
    "delivery_id": COL_DELIVERY_ID,
    "order_id": COL_DELIVERY_ORDER_ID,
    "carrier": COL_DELIVERY_CARRIER,
    "service_level": COL_DELIVERY_SERVICE_LEVEL,
    "status": COL_DELIVERY_STATUS,
    "shipped_at": COL_DELIVERY_SHIPPED_AT,
    "delivered_at": COL_DELIVERY_DELIVERED_AT,
    "cost": COL_DELIVERY_COST,
    "tracking_code": COL_DELIVERY_TRACKING,
    "destination": COL_DELIVERY_DESTINATION
}

print("Mapeamento de entregas:")

for campo_logico, campo_origem in mapeamento_entregas.items():
    print(
        f"{campo_logico:<20} -> "
        f"{campo_origem or 'não disponível na origem'}"
    )

In [0]:
# ============================================================
# REFERÊNCIA DE PEDIDOS
# ============================================================

pedidos_validos_entregas = (
    spark.table(
        f"`{CATALOG}`."
        f"`{SCHEMA_SILVER}`."
        f"`silver_pedidos`"
    )
    .select("order_id")
    .distinct()
    .withColumn(
        "_order_exists",
        F.lit(True)
    )
)

# Retorna a coluna encontrada ou NULL tipado quando o atributo não existe na origem
def campo_ou_nulo(
    caminho_coluna,
    tipo="string"
):
    if caminho_coluna:
        return F.col(caminho_coluna)

    return F.lit(None).cast(tipo)

In [0]:
# ============================================================
# PREPARAÇÃO SILVER — ENTREGAS
# ============================================================

silver_entregas_base = (
    bronze_entregas
    .select(
        F.col(COL_DELIVERY_ID).alias(
            "delivery_id_original"
        ),
        F.col(COL_DELIVERY_ORDER_ID).alias(
            "order_id_original"
        ),
        F.col(COL_DELIVERY_CARRIER).alias(
            "carrier_original"
        ),
        campo_ou_nulo(
            COL_DELIVERY_SERVICE_LEVEL
        ).alias(
            "service_level_original"
        ),
        F.col(COL_DELIVERY_STATUS).alias(
            "delivery_status_original"
        ),
        F.col(COL_DELIVERY_SHIPPED_AT).alias(
            "shipped_at_original"
        ),
        F.col(COL_DELIVERY_DELIVERED_AT).alias(
            "delivered_at_original"
        ),
        F.col(COL_DELIVERY_COST).alias(
            "delivery_cost_original"
        ),
        campo_ou_nulo(
            COL_DELIVERY_TRACKING
        ).alias(
            "tracking_code_original"
        ),
        campo_ou_nulo(
            COL_DELIVERY_DESTINATION
        ).cast("string").alias(
            "destination_original"
        ),
        F.col("_source_record_hash"),
        F.col("_source_file"),
        F.col("_source_path"),
        F.col("_ingestion_batch_id"),
        F.col("_ingestion_timestamp_utc")
    )
    .withColumn(
        "delivery_id",
        normalizar_id(
            F.col("delivery_id_original")
        )
    )
    .withColumn(
        "order_id",
        normalizar_id(
            F.col("order_id_original")
        )
    )
    .withColumn(
        "carrier",
        normalizar_dominio(
            F.col("carrier_original")
        )
    )
    .withColumn(
        "service_level",
        normalizar_dominio(
            F.col("service_level_original")
        )
    )
    .withColumn(
        "delivery_status",
        normalizar_dominio(
            F.col("delivery_status_original")
        )
    )
    .withColumn(
        "shipped_at",
        converter_timestamp_flexivel_coluna(
            F.col("shipped_at_original")
        )
    )
    .withColumn(
        "delivered_at",
        converter_timestamp_flexivel_coluna(
            F.col("delivered_at_original")
        )
    )
    .withColumn(
        "delivery_cost",
        converter_decimal_flexivel_coluna(
            F.col("delivery_cost_original"),
            "decimal(18,2)"
        )
    )
    .withColumn(
        "tracking_code",
        normalizar_id(
            F.col("tracking_code_original")
        )
    )
    .join(
        pedidos_validos_entregas,
        on="order_id",
        how="left"
    )
)

print(
    f"Entregas preparadas: "
    f"{silver_entregas_base.count()}"
)

display(
    silver_entregas_base
    .orderBy("delivery_id")
    .limit(10)
)

In [0]:
# ============================================================
# MÉTRICAS DE DUPLICIDADE — ENTREGAS
# ============================================================

metricas_entregas = (
    silver_entregas_base
    .groupBy("delivery_id")
    .agg(
        F.count("*").alias(
            "_records_by_key"
        ),
        F.countDistinct(
            F.coalesce(
                F.col("order_id"),
                F.lit("__NULL__")
            )
        ).alias(
            "_distinct_orders"
        ),
        F.countDistinct(
            F.coalesce(
                F.col("carrier"),
                F.lit("__NULL__")
            )
        ).alias(
            "_distinct_carriers"
        ),
        F.countDistinct(
            F.coalesce(
                F.col("delivery_status"),
                F.lit("__NULL__")
            )
        ).alias(
            "_distinct_statuses"
        ),
        F.countDistinct(
            F.coalesce(
                F.col("shipped_at").cast("string"),
                F.lit("__NULL__")
            )
        ).alias(
            "_distinct_shipped_dates"
        ),
        F.countDistinct(
            F.coalesce(
                F.col("delivered_at").cast("string"),
                F.lit("__NULL__")
            )
        ).alias(
            "_distinct_delivered_dates"
        ),
        F.countDistinct(
            F.coalesce(
                F.col("delivery_cost").cast("string"),
                F.lit("__NULL__")
            )
        ).alias(
            "_distinct_costs"
        )
    )
)

print("Entregas com chave duplicada:")

display(
    metricas_entregas
    .filter(
        F.col("_records_by_key") > 1
    )
    .orderBy("delivery_id")
)

In [0]:
# ============================================================
# QUALIDADE — ENTREGAS
# ============================================================

STATUS_ENTREGA_VALIDOS = [
    "PENDING",
    "PENDENTE",
    "SHIPPED",
    "ENVIADO",
    "IN_TRANSIT",
    "EM_TRANSITO",
    "OUT_FOR_DELIVERY",
    "SAIU_PARA_ENTREGA",
    "DELIVERED",
    "ENTREGUE",
    "CANCELLED",
    "CANCELED",
    "CANCELADO",
    "RETURNED",
    "DEVOLVIDO"
]

STATUS_ENTREGUE = [
    "DELIVERED",
    "ENTREGUE"
]

STATUS_EM_TRANSITO = [
    "IN_TRANSIT",
    "EM_TRANSITO"
]

silver_entregas_preparacao = (
    silver_entregas_base
    .join(
        metricas_entregas,
        on="delivery_id",
        how="left"
    )

    # --------------------------------------------------------
    # Chave da entrega
    # --------------------------------------------------------
    .withColumn(
        "dq_missing_delivery_id",
        F.col("delivery_id").isNull()
        | (F.col("delivery_id") == "")
    )

    # --------------------------------------------------------
    # Pedido relacionado
    # --------------------------------------------------------
    .withColumn(
        "dq_missing_order_id",
        F.col("order_id").isNull()
        | (F.col("order_id") == "")
    )
    .withColumn(
        "dq_orphan_order",
        F.col("order_id").isNotNull()
        & (F.col("order_id") != "")
        & F.col("_order_exists").isNull()
    )

    # --------------------------------------------------------
    # Transportadora
    # --------------------------------------------------------
    .withColumn(
        "dq_missing_carrier",
        F.col("carrier").isNull()
        | (F.col("carrier") == "")
    )

    # --------------------------------------------------------
    # Status
    # --------------------------------------------------------
    .withColumn(
        "dq_missing_status",
        F.col("delivery_status").isNull()
        | (F.col("delivery_status") == "")
    )
    .withColumn(
        "dq_unexpected_status",
        ~F.col("dq_missing_status")
        & (
            ~F.col("delivery_status").isin(
                STATUS_ENTREGA_VALIDOS
            )
        )
    )

    # --------------------------------------------------------
    # Data de envio
    # --------------------------------------------------------
    .withColumn(
        "dq_missing_shipped_at",
        F.col("shipped_at_original").isNull()
        | (
            F.trim(
                F.col("shipped_at_original").cast("string")
            ) == ""
        )
    )
    .withColumn(
        "dq_invalid_shipped_at",
        ~F.col("dq_missing_shipped_at")
        & F.col("shipped_at").isNull()
    )

    # --------------------------------------------------------
    # Data de entrega
    # --------------------------------------------------------
    .withColumn(
        "dq_missing_delivered_at",
        F.col("delivered_at_original").isNull()
        | (
            F.trim(
                F.col("delivered_at_original").cast("string")
            ) == ""
        )
    )
    .withColumn(
        "dq_invalid_delivered_at",
        ~F.col("dq_missing_delivered_at")
        & F.col("delivered_at").isNull()
    )
    .withColumn(
        "dq_delivered_before_shipped",
        F.col("shipped_at").isNotNull()
        & F.col("delivered_at").isNotNull()
        & (
            F.col("delivered_at")
            < F.col("shipped_at")
        )
    )

    # --------------------------------------------------------
    # Consistência entre status e datas
    # --------------------------------------------------------
    .withColumn(
        "dq_delivered_without_delivered_at",
        F.col("delivery_status").isin(
            STATUS_ENTREGUE
        )
        & F.col("delivered_at").isNull()
    )
    .withColumn(
        "dq_in_transit_with_delivered_at",
        F.col("delivery_status").isin(
            STATUS_EM_TRANSITO
        )
        & F.col("delivered_at").isNotNull()
    )

    # --------------------------------------------------------
    # Custo
    # --------------------------------------------------------
    .withColumn(
        "dq_missing_delivery_cost",
        F.col("delivery_cost_original").isNull()
        | (
            F.trim(
                F.col("delivery_cost_original").cast("string")
            ) == ""
        )
    )
    .withColumn(
        "dq_invalid_delivery_cost",
        ~F.col("dq_missing_delivery_cost")
        & F.col("delivery_cost").isNull()
    )
    .withColumn(
        "dq_negative_delivery_cost",
        F.col("delivery_cost").isNotNull()
        & (F.col("delivery_cost") < 0)
    )

    # --------------------------------------------------------
    # Disponibilidade dos campos opcionais na origem
    # --------------------------------------------------------
    .withColumn(
        "source_has_service_level",
        F.lit(
            COL_DELIVERY_SERVICE_LEVEL is not None
        )
    )
    .withColumn(
        "source_has_tracking_code",
        F.lit(
            COL_DELIVERY_TRACKING is not None
        )
    )

    # --------------------------------------------------------
    # Duplicidade
    # --------------------------------------------------------
    .withColumn(
        "dq_duplicate_key",
        F.col("_records_by_key") > 1
    )
    .withColumn(
        "dq_conflicting_duplicate",
        (
            F.col("_distinct_orders") > 1
        )
        |
        (
            F.col("_distinct_carriers") > 1
        )
        |
        (
            F.col("_distinct_statuses") > 1
        )
        |
        (
            F.col("_distinct_shipped_dates") > 1
        )
        |
        (
            F.col("_distinct_delivered_dates") > 1
        )
        |
        (
            F.col("_distinct_costs") > 1
        )
    )
)

In [0]:
# ============================================================
# PONTUAÇÃO DE QUALIDADE — ENTREGAS
# ============================================================

silver_entregas_preparacao = (
    silver_entregas_preparacao
    .withColumn(
        "_quality_score",

        # Identificador da entrega
        F.when(
            ~F.col("dq_missing_delivery_id"),
            20
        ).otherwise(0)

        +

        # Pedido existente
        F.when(
            F.col("_order_exists") == True,
            25
        ).otherwise(0)

        +

        # Transportadora
        F.when(
            ~F.col("dq_missing_carrier"),
            10
        ).otherwise(0)

        +

        # Status reconhecido
        F.when(
            ~F.col("dq_missing_status")
            & ~F.col("dq_unexpected_status"),
            10
        ).otherwise(0)

        +

        # Data de envio válida
        F.when(
            ~F.col("dq_missing_shipped_at")
            & ~F.col("dq_invalid_shipped_at"),
            10
        ).otherwise(0)

        +

        # Data de entrega válida quando informada
        F.when(
            ~F.col("dq_invalid_delivered_at"),
            5
        ).otherwise(0)

        +

        # Ordem temporal coerente
        F.when(
            ~F.col("dq_delivered_before_shipped"),
            10
        ).otherwise(0)

        +

        # Coerência entre status e datas
        F.when(
            ~F.col("dq_delivered_without_delivered_at")
            & ~F.col("dq_in_transit_with_delivered_at"),
            15
        ).otherwise(0)

        +

        # Custo válido
        F.when(
            ~F.col("dq_missing_delivery_cost")
            & ~F.col("dq_invalid_delivery_cost")
            & ~F.col("dq_negative_delivery_cost"),
            10
        ).otherwise(0)
    )
)

In [0]:
# ============================================================
# DEDUPLICAÇÃO — ENTREGAS
# ============================================================

janela_prioridade_entrega = (
    Window
    .partitionBy("delivery_id")
    .orderBy(
        F.col("_quality_score").desc(),
        F.col("delivered_at").desc_nulls_last(),
        F.col("shipped_at").desc_nulls_last(),
        F.col(
            "_ingestion_timestamp_utc"
        ).desc(),
        F.col(
            "_source_record_hash"
        ).asc()
    )
)

silver_entregas = (
    silver_entregas_preparacao
    .withColumn(
        "_deduplication_rank",
        F.row_number().over(
            janela_prioridade_entrega
        )
    )
    .filter(
        F.col("_deduplication_rank") == 1
    )
    .withColumn(
        "_silver_fact_run_id",
        F.lit(SILVER_FACT_RUN_ID)
    )
    .withColumn(
        "_silver_processed_at_utc",
        F.lit(SILVER_FACT_TIMESTAMP)
    )
    .drop(
        "_order_exists",
        "_records_by_key",
        "_distinct_orders",
        "_distinct_carriers",
        "_distinct_statuses",
        "_distinct_shipped_dates",
        "_distinct_delivered_dates",
        "_distinct_costs",
        "_quality_score",
        "_deduplication_rank"
    )
)

display(
    silver_entregas
    .orderBy("delivery_id")
)

In [0]:
# ============================================================
# VALIDAÇÃO — ENTREGAS
# ============================================================

quantidade_entregas_bronze = (
    bronze_entregas.count()
)

quantidade_entregas_distintas_esperada = (
    bronze_entregas
    .select(
        normalizar_id(
            F.col(COL_DELIVERY_ID)
        ).alias("delivery_id")
    )
    .distinct()
    .count()
)

quantidade_entregas_silver = (
    silver_entregas.count()
)

duplicidades_entregas_silver = (
    silver_entregas
    .groupBy("delivery_id")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

chaves_nulas_entregas = (
    silver_entregas
    .filter(
        F.col("delivery_id").isNull()
        | (F.col("delivery_id") == "")
    )
    .count()
)

entregas_pedidos_orfaos = (
    silver_entregas
    .filter(
        F.col("dq_orphan_order") == True
    )
    .count()
)

entregas_envio_invalido = (
    silver_entregas
    .filter(
        F.col("dq_invalid_shipped_at") == True
    )
    .count()
)

entregas_custo_invalido = (
    silver_entregas
    .filter(
        F.col("dq_invalid_delivery_cost") == True
    )
    .count()
)

entregas_status_data_inconsistentes = (
    silver_entregas
    .filter(
        (
            F.col(
                "dq_delivered_without_delivered_at"
            ) == True
        )
        |
        (
            F.col(
                "dq_in_transit_with_delivered_at"
            ) == True
        )
        |
        (
            F.col(
                "dq_delivered_before_shipped"
            ) == True
        )
    )
    .count()
)

print(
    f"Bronze: "
    f"{quantidade_entregas_bronze}"
)

print(
    f"Chaves distintas esperadas: "
    f"{quantidade_entregas_distintas_esperada}"
)

print(
    f"Silver: "
    f"{quantidade_entregas_silver}"
)

print(
    f"Registros consolidados: "
    f"{quantidade_entregas_bronze - quantidade_entregas_silver}"
)

print(
    f"Duplicidades na Silver: "
    f"{duplicidades_entregas_silver}"
)

print(
    f"Chaves nulas na Silver: "
    f"{chaves_nulas_entregas}"
)

print(
    f"Entregas com pedido órfão: "
    f"{entregas_pedidos_orfaos}"
)

print(
    f"Datas de envio inválidas: "
    f"{entregas_envio_invalido}"
)

print(
    f"Custos inválidos: "
    f"{entregas_custo_invalido}"
)

print(
    f"Inconsistências status × data: "
    f"{entregas_status_data_inconsistentes}"
)

if (
    quantidade_entregas_silver
    != quantidade_entregas_distintas_esperada
):
    raise RuntimeError(
        "A Silver de entregas não corresponde as chaves distintas da Bronze"
    )

if duplicidades_entregas_silver > 0:
    raise RuntimeError(
        "A Silver de entregas ainda possui duplicidades!"
    )

if chaves_nulas_entregas > 0:
    raise RuntimeError(
        "A Silver de entregas possui chaves nulas"
    )

In [0]:
# ============================================================
# CONFERÊNCIA DE QUALIDADE — ENTREGAS
# ============================================================

display(
    silver_entregas
    .filter(
        F.col("dq_orphan_order")
        | F.col("dq_invalid_shipped_at")
        | F.col("dq_invalid_delivery_cost")
        | F.col("dq_delivered_without_delivered_at")
        | F.col("dq_in_transit_with_delivered_at")
        | F.col("dq_delivered_before_shipped")
    )
    .select(
        "delivery_id",
        "order_id",
        "carrier",
        "delivery_status",
        "shipped_at_original",
        "shipped_at",
        "delivered_at_original",
        "delivered_at",
        "delivery_cost_original",
        "delivery_cost",
        "dq_orphan_order",
        "dq_invalid_shipped_at",
        "dq_invalid_delivery_cost",
        "dq_delivered_without_delivered_at",
        "dq_in_transit_with_delivered_at",
        "dq_delivered_before_shipped"
    )
    .orderBy("delivery_id")
)

In [0]:
# ============================================================
# SILVER — ENTREGAS
# ============================================================

TABELA_SILVER_ENTREGAS = (
    f"`{CATALOG}`."
    f"`{SCHEMA_SILVER}`."
    f"`silver_entregas`"
)

(
    silver_entregas.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        TABELA_SILVER_ENTREGAS
    )
)

quantidade_entregas_persistidas = (
    spark.table(
        TABELA_SILVER_ENTREGAS
    )
    .count()
)

if (
    quantidade_entregas_persistidas
    != quantidade_entregas_silver
):
    raise RuntimeError(
        "A quantidade persistida de entregas diverge da quantidade do DataFrame Silver"
    )

print(
    f"Tabela criada: "
    f"{CATALOG}.{SCHEMA_SILVER}.silver_entregas"
)

print(
    f"Registros persistidos: "
    f"{quantidade_entregas_persistidas}"
)

In [0]:
# ============================================================
# RESOLUÇÃO DO ESQUEMA — OCORRÊNCIAS
# ============================================================

COL_OCCURRENCE_ID = localizar_campo_spark(
    bronze_ocorrencias,
    [
        "occurrence_id",
        "ocorrencia_id",
        "ticket_id",
        "case_id",
        "protocol_id",
        "protocolo",
        "id"
    ],
    "identificador da ocorrência"
)

COL_OCCURRENCE_ORDER_ID = localizar_campo_spark(
    bronze_ocorrencias,
    [
        "order_ref",
        "order_id",
        "pedido_id",
        "id_pedido",
        "order.order_id"
    ],
    "identificador do pedido"
)

COL_OCCURRENCE_STATUS = localizar_campo_spark(
    bronze_ocorrencias,
    [
        "status",
        "occurrence_status",
        "ticket_status",
        "situacao"
    ],
    "status da ocorrência",
    obrigatorio=False
)

COL_OCCURRENCE_CATEGORY = localizar_campo_spark(
    bronze_ocorrencias,
    [
        "category",
        "occurrence_type",
        "type",
        "tipo",
        "categoria"
    ],
    "categoria",
    obrigatorio=False
)

COL_OCCURRENCE_SUBCATEGORY = localizar_campo_spark(
    bronze_ocorrencias,
    [
        "subcategory",
        "subtype",
        "subcategoria"
    ],
    "subcategoria",
    obrigatorio=False
)

COL_OCCURRENCE_PRIORITY = localizar_campo_spark(
    bronze_ocorrencias,
    [
        "priority",
        "severity",
        "criticity",
        "prioridade"
    ],
    "prioridade",
    obrigatorio=False
)

COL_OCCURRENCE_CHANNEL = localizar_campo_spark(
    bronze_ocorrencias,
    [
        "channel",
        "contact_channel",
        "service_channel",
        "canal"
    ],
    "canal de atendimento",
    obrigatorio=False
)

COL_OCCURRENCE_OPENED_AT = localizar_campo_spark(
    bronze_ocorrencias,
    [
        "opened_at",
        "created_at",
        "creation_date",
        "open_date",
        "data_abertura",
        "dt_abertura",
        "timestamps.opened_at"
    ],
    "data de abertura",
    obrigatorio=False
)

COL_OCCURRENCE_CLOSED_AT = localizar_campo_spark(
    bronze_ocorrencias,
    [
        "closed_at",
        "resolved_at",
        "resolution_date",
        "close_date",
        "data_fechamento",
        "dt_fechamento",
        "timestamps.closed_at"
    ],
    "data de fechamento",
    obrigatorio=False
)

COL_OCCURRENCE_DESCRIPTION = localizar_campo_spark(
    bronze_ocorrencias,
    [
        "description",
        "summary",
        "subject",
        "descricao",
        "motivo"
    ],
    "descrição",
    obrigatorio=False
)

mapeamento_ocorrencias = {
    "occurrence_id": COL_OCCURRENCE_ID,
    "order_id": COL_OCCURRENCE_ORDER_ID,
    "status": COL_OCCURRENCE_STATUS,
    "category": COL_OCCURRENCE_CATEGORY,
    "subcategory": COL_OCCURRENCE_SUBCATEGORY,
    "priority": COL_OCCURRENCE_PRIORITY,
    "channel": COL_OCCURRENCE_CHANNEL,
    "opened_at": COL_OCCURRENCE_OPENED_AT,
    "closed_at": COL_OCCURRENCE_CLOSED_AT,
    "description": COL_OCCURRENCE_DESCRIPTION
}

print("Mapeamento de ocorrências:")

for campo_logico, campo_origem in mapeamento_ocorrencias.items():
    print(
        f"{campo_logico:<20} -> "
        f"{campo_origem or 'não disponível na origem'}"
    )

In [0]:
# ============================================================
# REFERÊNCIA DE PEDIDOS
# ============================================================

pedidos_validos_ocorrencias = (
    spark.table(
        f"`{CATALOG}`."
        f"`{SCHEMA_SILVER}`."
        f"`silver_pedidos`"
    )
    .select("order_id")
    .distinct()
    .withColumn(
        "_order_exists",
        F.lit(True)
    )
)

In [0]:
# ============================================================
# PREPARAÇÃO SILVER — OCORRÊNCIAS
# ============================================================

silver_ocorrencias_base = (
    bronze_ocorrencias
    .select(
        F.col(COL_OCCURRENCE_ID).alias(
            "occurrence_id_original"
        ),
        F.col(COL_OCCURRENCE_ORDER_ID).alias(
            "order_id_original"
        ),
        campo_ou_nulo(
            COL_OCCURRENCE_STATUS
        ).alias(
            "occurrence_status_original"
        ),
        campo_ou_nulo(
            COL_OCCURRENCE_CATEGORY
        ).alias(
            "category_original"
        ),
        campo_ou_nulo(
            COL_OCCURRENCE_SUBCATEGORY
        ).alias(
            "subcategory_original"
        ),
        campo_ou_nulo(
            COL_OCCURRENCE_PRIORITY
        ).alias(
            "priority_original"
        ),
        campo_ou_nulo(
            COL_OCCURRENCE_CHANNEL
        ).alias(
            "service_channel_original"
        ),
        campo_ou_nulo(
            COL_OCCURRENCE_OPENED_AT
        ).alias(
            "opened_at_original"
        ),
        campo_ou_nulo(
            COL_OCCURRENCE_CLOSED_AT
        ).alias(
            "closed_at_original"
        ),
        campo_ou_nulo(
            COL_OCCURRENCE_DESCRIPTION
        ).alias(
            "description_original"
        ),
        F.col("_source_record_hash"),
        F.col("_source_file"),
        F.col("_source_path"),
        F.col("_ingestion_batch_id"),
        F.col("_ingestion_timestamp_utc")
    )
    .withColumn(
        "occurrence_id",
        normalizar_id(
            F.col("occurrence_id_original")
        )
    )
    .withColumn(
        "order_id",
        normalizar_id(
            F.col("order_id_original")
        )
    )
    .withColumn(
        "occurrence_status",
        normalizar_dominio(
            F.col("occurrence_status_original")
        )
    )
    .withColumn(
        "category",
        normalizar_dominio(
            F.col("category_original")
        )
    )
    .withColumn(
        "subcategory",
        normalizar_dominio(
            F.col("subcategory_original")
        )
    )
    .withColumn(
        "priority",
        normalizar_dominio(
            F.col("priority_original")
        )
    )
    .withColumn(
        "service_channel",
        normalizar_dominio(
            F.col("service_channel_original")
        )
    )
    .withColumn(
        "opened_at",
        converter_timestamp_flexivel_coluna(
            F.col("opened_at_original")
        )
    )
    .withColumn(
        "closed_at",
        converter_timestamp_flexivel_coluna(
            F.col("closed_at_original")
        )
    )
    .withColumn(
        "description",
        F.trim(
            F.col("description_original").cast("string")
        )
    )
    .withColumn(
        "resolution_hours",
        F.when(
            F.col("opened_at").isNotNull()
            & F.col("closed_at").isNotNull(),
            F.round(
                (
                    F.col("closed_at").cast("long")
                    - F.col("opened_at").cast("long")
                ) / F.lit(3600.0),
                2
            )
        )
    )
    .join(
        pedidos_validos_ocorrencias,
        on="order_id",
        how="left"
    )
)

print(
    f"Ocorrências preparadas: "
    f"{silver_ocorrencias_base.count()}"
)

display(
    silver_ocorrencias_base
    .orderBy("occurrence_id")
    .limit(10)
)

In [0]:
# ============================================================
# MÉTRICAS DE DUPLICIDADE — OCORRÊNCIAS
# ============================================================

metricas_ocorrencias = (
    silver_ocorrencias_base
    .groupBy("occurrence_id")
    .agg(
        F.count("*").alias(
            "_records_by_key"
        ),
        F.countDistinct(
            F.coalesce(
                F.col("order_id"),
                F.lit("__NULL__")
            )
        ).alias(
            "_distinct_orders"
        ),
        F.countDistinct(
            F.coalesce(
                F.col("occurrence_status"),
                F.lit("__NULL__")
            )
        ).alias(
            "_distinct_statuses"
        ),
        F.countDistinct(
            F.coalesce(
                F.col("category"),
                F.lit("__NULL__")
            )
        ).alias(
            "_distinct_categories"
        ),
        F.countDistinct(
            F.coalesce(
                F.col("opened_at").cast("string"),
                F.lit("__NULL__")
            )
        ).alias(
            "_distinct_opened_dates"
        ),
        F.countDistinct(
            F.coalesce(
                F.col("closed_at").cast("string"),
                F.lit("__NULL__")
            )
        ).alias(
            "_distinct_closed_dates"
        )
    )
)

display(
    metricas_ocorrencias
    .filter(
        F.col("_records_by_key") > 1
    )
    .orderBy("occurrence_id")
)

In [0]:
# ============================================================
# QUALIDADE — OCORRÊNCIAS
# ============================================================

silver_ocorrencias_preparacao = (
    silver_ocorrencias_base
    .join(
        metricas_ocorrencias,
        on="occurrence_id",
        how="left"
    )
    .withColumn(
        "dq_missing_occurrence_id",
        F.col("occurrence_id").isNull()
        | (F.col("occurrence_id") == "")
    )
    .withColumn(
        "dq_missing_order_id",
        F.col("order_id").isNull()
        | (F.col("order_id") == "")
    )
    .withColumn(
        "dq_orphan_order",
        F.col("order_id").isNotNull()
        & (F.col("order_id") != "")
        & F.col("_order_exists").isNull()
    )
    .withColumn(
        "dq_missing_status",
        F.col("occurrence_status").isNull()
        | (F.col("occurrence_status") == "")
    )
    .withColumn(
        "dq_missing_category",
        F.col("category").isNull()
        | (F.col("category") == "")
    )
    .withColumn(
        "dq_missing_opened_at",
        F.col("opened_at_original").isNull()
        | (
            F.trim(
                F.col("opened_at_original").cast("string")
            ) == ""
        )
    )
    .withColumn(
        "dq_invalid_opened_at",
        ~F.col("dq_missing_opened_at")
        & F.col("opened_at").isNull()
    )
    .withColumn(
        "dq_missing_closed_at",
        F.col("closed_at_original").isNull()
        | (
            F.trim(
                F.col("closed_at_original").cast("string")
            ) == ""
        )
    )
    .withColumn(
        "dq_invalid_closed_at",
        ~F.col("dq_missing_closed_at")
        & F.col("closed_at").isNull()
    )
    .withColumn(
        "dq_closed_before_opened",
        F.col("opened_at").isNotNull()
        & F.col("closed_at").isNotNull()
        & (
            F.col("closed_at")
            < F.col("opened_at")
        )
    )
    .withColumn(
        "dq_negative_resolution_hours",
        F.col("resolution_hours").isNotNull()
        & (
            F.col("resolution_hours") < 0
        )
    )
    .withColumn(
        "source_has_status",
        F.lit(
            COL_OCCURRENCE_STATUS is not None
        )
    )
    .withColumn(
        "source_has_category",
        F.lit(
            COL_OCCURRENCE_CATEGORY is not None
        )
    )
    .withColumn(
        "source_has_opened_at",
        F.lit(
            COL_OCCURRENCE_OPENED_AT is not None
        )
    )
    .withColumn(
        "source_has_closed_at",
        F.lit(
            COL_OCCURRENCE_CLOSED_AT is not None
        )
    )
    .withColumn(
        "dq_duplicate_key",
        F.col("_records_by_key") > 1
    )
    .withColumn(
        "dq_conflicting_duplicate",
        (
            F.col("_distinct_orders") > 1
        )
        |
        (
            F.col("_distinct_statuses") > 1
        )
        |
        (
            F.col("_distinct_categories") > 1
        )
        |
        (
            F.col("_distinct_opened_dates") > 1
        )
        |
        (
            F.col("_distinct_closed_dates") > 1
        )
    )
)

In [0]:
# ============================================================
# PONTUAÇÃO DE QUALIDADE — OCORRÊNCIAS
# ============================================================

silver_ocorrencias_preparacao = (
    silver_ocorrencias_preparacao
    .withColumn(
        "_quality_score",
        F.when(
            ~F.col("dq_missing_occurrence_id"),
            25
        ).otherwise(0)
        +
        F.when(
            F.col("_order_exists") == True,
            25
        ).otherwise(0)
        +
        F.when(
            ~F.col("dq_missing_status"),
            10
        ).otherwise(0)
        +
        F.when(
            ~F.col("dq_missing_category"),
            10
        ).otherwise(0)
        +
        F.when(
            ~F.col("dq_missing_opened_at")
            & ~F.col("dq_invalid_opened_at"),
            15
        ).otherwise(0)
        +
        F.when(
            ~F.col("dq_invalid_closed_at"),
            5
        ).otherwise(0)
        +
        F.when(
            ~F.col("dq_closed_before_opened"),
            10
        ).otherwise(0)
        +
        F.when(
            F.col("description").isNotNull()
            & (F.col("description") != ""),
            5
        ).otherwise(0)
    )
)

janela_prioridade_ocorrencia = (
    Window
    .partitionBy("occurrence_id")
    .orderBy(
        F.col("_quality_score").desc(),
        F.col("closed_at").desc_nulls_last(),
        F.col("opened_at").desc_nulls_last(),
        F.col(
            "_ingestion_timestamp_utc"
        ).desc(),
        F.col(
            "_source_record_hash"
        ).asc()
    )
)

silver_ocorrencias = (
    silver_ocorrencias_preparacao
    .withColumn(
        "_deduplication_rank",
        F.row_number().over(
            janela_prioridade_ocorrencia
        )
    )
    .filter(
        F.col("_deduplication_rank") == 1
    )
    .withColumn(
        "_silver_fact_run_id",
        F.lit(SILVER_FACT_RUN_ID)
    )
    .withColumn(
        "_silver_processed_at_utc",
        F.lit(SILVER_FACT_TIMESTAMP)
    )
    .drop(
        "_order_exists",
        "_records_by_key",
        "_distinct_orders",
        "_distinct_statuses",
        "_distinct_categories",
        "_distinct_opened_dates",
        "_distinct_closed_dates",
        "_quality_score",
        "_deduplication_rank"
    )
)

display(
    silver_ocorrencias
    .orderBy("occurrence_id")
)

In [0]:
# ============================================================
# VALIDAÇÃO — OCORRÊNCIAS
# ============================================================

quantidade_ocorrencias_bronze = (
    bronze_ocorrencias.count()
)

quantidade_ocorrencias_distintas_esperada = (
    bronze_ocorrencias
    .select(
        normalizar_id(
            F.col(COL_OCCURRENCE_ID)
        ).alias("occurrence_id")
    )
    .distinct()
    .count()
)

quantidade_ocorrencias_silver = (
    silver_ocorrencias.count()
)

duplicidades_ocorrencias_silver = (
    silver_ocorrencias
    .groupBy("occurrence_id")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

chaves_nulas_ocorrencias = (
    silver_ocorrencias
    .filter(
        F.col("occurrence_id").isNull()
        | (F.col("occurrence_id") == "")
    )
    .count()
)

ocorrencias_pedidos_orfaos = (
    silver_ocorrencias
    .filter(
        F.col("dq_orphan_order") == True
    )
    .count()
)

ocorrencias_datas_inconsistentes = (
    silver_ocorrencias
    .filter(
        F.col("dq_invalid_opened_at")
        | F.col("dq_invalid_closed_at")
        | F.col("dq_closed_before_opened")
    )
    .count()
)

print(f"Bronze: {quantidade_ocorrencias_bronze}")

print(
    f"Chaves distintas esperadas: "
    f"{quantidade_ocorrencias_distintas_esperada}"
)

print(f"Silver: {quantidade_ocorrencias_silver}")

print(
    f"Registros consolidados: "
    f"{quantidade_ocorrencias_bronze - quantidade_ocorrencias_silver}"
)

print(
    f"Duplicidades na Silver: "
    f"{duplicidades_ocorrencias_silver}"
)

print(
    f"Chaves nulas na Silver: "
    f"{chaves_nulas_ocorrencias}"
)

print(
    f"Ocorrências com pedido órfão: "
    f"{ocorrencias_pedidos_orfaos}"
)

print(
    f"Ocorrências com datas inconsistentes: "
    f"{ocorrencias_datas_inconsistentes}"
)

if (
    quantidade_ocorrencias_silver
    != quantidade_ocorrencias_distintas_esperada
):
    raise RuntimeError(
        "A Silver de ocorrências não corresponde as chaves distintas da Bronze"
    )

if duplicidades_ocorrencias_silver > 0:
    raise RuntimeError(
        "A Silver de ocorrências possui duplicidades"
    )

if chaves_nulas_ocorrencias > 0:
    raise RuntimeError(
        "A Silver de ocorrências possui chaves nulas"
    )

In [0]:
# ============================================================
# SILVER — OCORRÊNCIAS
# ============================================================

TABELA_SILVER_OCORRENCIAS = (
    f"`{CATALOG}`."
    f"`{SCHEMA_SILVER}`."
    f"`silver_ocorrencias`"
)

(
    silver_ocorrencias.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        TABELA_SILVER_OCORRENCIAS
    )
)

quantidade_ocorrencias_persistidas = (
    spark.table(
        TABELA_SILVER_OCORRENCIAS
    )
    .count()
)

if (
    quantidade_ocorrencias_persistidas
    != quantidade_ocorrencias_silver
):
    raise RuntimeError(
        "A persistência de ocorrências apresentou divergência!"
    )

print(
    f"Tabela criada: "
    f"{CATALOG}.{SCHEMA_SILVER}.silver_ocorrencias"
)

print(
    f"Registros persistidos: "
    f"{quantidade_ocorrencias_persistidas}"
)

In [0]:
# ============================================================
# AUDITORIA CONSOLIDADA DOS FATOS SILVER
# ============================================================

from functools import reduce


def contar_registros_com_dq(dataframe):
    colunas_dq = [
        coluna
        for coluna in dataframe.columns
        if coluna.startswith("dq_")
    ]

    if not colunas_dq:
        return 0

    condicao = reduce(
        lambda anterior, atual: anterior | atual,
        [
            F.coalesce(
                F.col(coluna),
                F.lit(False)
            )
            for coluna in colunas_dq
        ]
    )

    return dataframe.filter(condicao).count()


def contar_orfaos(dataframe):
    colunas_orfas = [
        coluna
        for coluna in dataframe.columns
        if coluna.startswith("dq_orphan_")
    ]

    if not colunas_orfas:
        return 0

    condicao = reduce(
        lambda anterior, atual: anterior | atual,
        [
            F.coalesce(
                F.col(coluna),
                F.lit(False)
            )
            for coluna in colunas_orfas
        ]
    )

    return dataframe.filter(condicao).count()


configuracao_fatos = [
    (
        "silver_pedidos",
        bronze_pedidos,
        ["order_id"],
        "um registro por pedido"
    ),
    (
        "silver_pedidos_itens",
        bronze_itens,
        ["order_id", "item_seq"],
        "um registro por item do pedido"
    ),
    (
        "silver_entregas",
        bronze_entregas,
        ["delivery_id"],
        "um registro por entrega"
    ),
    (
        "silver_ocorrencias",
        bronze_ocorrencias,
        ["occurrence_id"],
        "um registro por ocorrência"
    )
]

resultado_auditoria_fatos = []

for (
    tabela,
    dataframe_bronze,
    chaves,
    granularidade
) in configuracao_fatos:

    dataframe_silver = spark.table(
        f"`{CATALOG}`."
        f"`{SCHEMA_SILVER}`."
        f"`{tabela}`"
    )

    quantidade_bronze = dataframe_bronze.count()
    quantidade_silver = dataframe_silver.count()

    duplicidades_saida = (
        dataframe_silver
        .groupBy(*chaves)
        .count()
        .filter(
            F.col("count") > 1
        )
        .count()
    )

    condicao_chave_nula = reduce(
        lambda anterior, atual: anterior | atual,
        [
            F.col(chave).isNull()
            | (
                F.col(chave).cast("string") == ""
            )
            for chave in chaves
        ]
    )

    chaves_nulas_saida = (
        dataframe_silver
        .filter(condicao_chave_nula)
        .count()
    )

    registros_com_dq = contar_registros_com_dq(
        dataframe_silver
    )

    registros_orfaos = contar_orfaos(
        dataframe_silver
    )

    status = (
        "SUCESSO"
        if (
            duplicidades_saida == 0
            and chaves_nulas_saida == 0
        )
        else "ERRO"
    )

    resultado_auditoria_fatos.append(
        (
            SILVER_FACT_RUN_ID,
            tabela,
            granularidade,
            quantidade_bronze,
            quantidade_silver,
            quantidade_bronze - quantidade_silver,
            duplicidades_saida,
            chaves_nulas_saida,
            registros_orfaos,
            registros_com_dq,
            status,
            SILVER_FACT_TIMESTAMP
        )
    )


schema_auditoria_fatos = """
    silver_fact_run_id STRING,
    tabela STRING,
    granularidade STRING,
    quantidade_bronze LONG,
    quantidade_silver LONG,
    registros_consolidados LONG,
    duplicidades_saida LONG,
    chaves_nulas_saida LONG,
    registros_orfaos LONG,
    registros_com_dq LONG,
    status STRING,
    processed_at_utc TIMESTAMP
"""

df_auditoria_fatos = (
    spark.createDataFrame(
        resultado_auditoria_fatos,
        schema=schema_auditoria_fatos
    )
    .orderBy("tabela")
)

display(df_auditoria_fatos)

In [0]:
# ============================================================
# AUDITORIA
# ============================================================

TABELA_AUDITORIA_FATOS = (
    f"`{CATALOG}`."
    f"`{SCHEMA_SILVER}`."
    f"`silver_fact_audit`"
)

(
    df_auditoria_fatos.write
    .format("delta")
    .mode("append")
    .option(
        "mergeSchema",
        "true"
    )
    .saveAsTable(
        TABELA_AUDITORIA_FATOS
    )
)

erros_estruturais = (
    df_auditoria_fatos
    .filter(
        F.col("status") != "SUCESSO"
    )
    .count()
)

if erros_estruturais > 0:
    raise RuntimeError(
        "A auditoria identificou erros estruturais nas tabelas Silver"
    )

print(
    "Fatos Silver concluídos e auditados com sucesso"
)

# Conclusão dos fatos Silver

Foram construídas e persistidas quatro tabelas transacionais:

- silver_pedidos: um registro por pedido
- silver_pedidos_itens: um registro por item do pedido
- silver_entregas: um registro por entrega
- silver_ocorrencias: um registro por ocorrência

As transformações incluíram:

- normalização de identificadores e domínios
- conversão segura de datas e valores
- resolução determinística de duplicidades
- validação de integridade referencial
- validação da consistência financeira dos pedidos e itens
- validação da consistência temporal das entregas e ocorrências
- preservação e sinalização de registros inválidos
- auditoria das quantidades, chaves e granularidades

Problemas de qualidade não críticos foram mantidos com indicadores `dq_*`, permitindo transparência e investigação